# Task 3: Data Quality Analysis: Adult (Census Income) Dataset

**Dataset:** UCI Adult, https://doi.org/10.24432/C5XW20 (file used: `adult.csv`)  
**Tools:** Python (pandas, NumPy, SciPy, Matplotlib, Seaborn). No spreadsheet software was used for EDA.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

os.makedirs('figures', exist_ok=True)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 0. Load data and first look

In [ ]:
df = pd.read_csv('adult.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
num_cols = ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week']
cat_cols = [c for c in df.columns if c not in num_cols and c != 'income']
print('Numeric    :', len(num_cols), num_cols)
print('Categorical:', len(cat_cols), cat_cols)

## 1. Missing values
A plain null check misses the placeholder `'?'`.

In [ ]:
print('Nulls detected by default isnull():', df.isnull().sum().sum())
print("Cells equal to '?':", (df == '?').sum().sum())

In [ ]:
df_clean = df.replace('?', np.nan)
missing = pd.DataFrame({
    'missing_count': df_clean.isna().sum(),
    'missing_pct': (df_clean.isna().mean() * 100).round(2)
})
missing = missing[missing.missing_count > 0].sort_values('missing_count', ascending=False)
missing

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(x=missing.missing_count, y=missing.index, color='#4C72B0', ax=ax)
ax.set_title('Missing values by column (after replacing "?")')
plt.tight_layout()
plt.savefig('figures/fig_missing.png', dpi=150)
plt.show()

In [ ]:
# Do workclass and occupation go missing together? (suggests MAR, not MCAR)
both = (df_clean.workclass.isna() & df_clean.occupation.isna()).sum()
print('Rows missing BOTH workclass and occupation:', both)
any_missing = df_clean.isna().any(axis=1)
print('Rows lost if all rows with any missing value are deleted:',
      any_missing.sum(), f'({any_missing.mean()*100:.1f}%)')

## 2. Duplicate records

In [ ]:
n_dup = df.duplicated().sum()
print('Exact duplicate rows:', n_dup, f'({n_dup/len(df)*100:.2f}%)')
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)

## 3. Outliers (IQR rule)

In [ ]:
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[c] < lo) | (df[c] > hi)).sum()
    rows.append([c, lo, hi, n, round(n / len(df) * 100, 2)])
outliers = pd.DataFrame(rows, columns=['feature', 'lower_bound', 'upper_bound', 'n_outliers', 'pct'])
outliers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.flat, num_cols):
    sns.boxplot(y=df[c], ax=ax, color='#8FBC8F')
    ax.set_title(c)
plt.tight_layout()
plt.savefig('figures/fig_boxplots.png', dpi=150)
plt.show()

In [ ]:
# Why are capital.gain / capital.loss flagged so heavily?
print('capital.gain == 0 :', round((df['capital.gain'] == 0).mean() * 100, 1), '%')
print('capital.loss == 0 :', round((df['capital.loss'] == 0).mean() * 100, 1), '%')
print('capital.gain == 99999 (possible top-coding):', (df['capital.gain'] == 99999).sum())

## 4. Feature distributions and skewness

In [ ]:
summary = df[num_cols].describe().T
summary['median'] = df[num_cols].median()
summary['skewness'] = df[num_cols].skew()
summary[['mean', 'median', 'std', 'min', 'max', 'skewness']].round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.flat, num_cols):
    sns.histplot(df[c], bins=40, ax=ax, color='#55A868')
    ax.set_title(f'{c} (skew = {df[c].skew():.2f})')
plt.tight_layout()
plt.savefig('figures/fig_histograms.png', dpi=150)
plt.show()

In [ ]:
# Categorical dominance
for c in ['native.country', 'workclass', 'race']:
    print(c)
    print((df[c].value_counts(normalize=True).head(3) * 100).round(1).to_string(), '\n')

## 5. Correlation among variables

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Pearson correlation (numeric features)')
plt.tight_layout()
plt.savefig('figures/fig_corr.png', dpi=150)
plt.show()

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

pairs = [('education', 'education.num'), ('sex', 'relationship'), ('marital.status', 'relationship')]
pd.DataFrame([[a, b, round(cramers_v(df[a], df[b]), 2)] for a, b in pairs],
             columns=['feature_1', 'feature_2', 'cramers_v'])

## 6. Class imbalance

In [ ]:
counts = df['income'].value_counts()
pct = (df['income'].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({'count': counts, 'pct': pct}))
print('Imbalance ratio:', round(counts.max() / counts.min(), 2), ': 1')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.countplot(x='income', data=df, hue='income', palette=['#4C72B0', '#C44E52'], legend=False, ax=ax)
ax.set_title('Income class distribution')
plt.tight_layout()
plt.savefig('figures/fig_imbalance.png', dpi=150)
plt.show()

## 7. Noise

In [ ]:
print('capital.gain max:', df['capital.gain'].max(), '| records at the cap:', (df['capital.gain'] == 99999).sum())
print('fnlwgt median:', df['fnlwgt'].median(), '| max:', df['fnlwgt'].max())

## 8. Data inconsistency

In [ ]:
# 1) education vs education.num should map 1-to-1
print('education levels mapping to >1 education.num:',
      (df.groupby('education')['education.num'].nunique() > 1).sum())

# 2) sex vs relationship
mismatch = df[((df.sex == 'Male') & (df.relationship == 'Wife')) |
              ((df.sex == 'Female') & (df.relationship == 'Husband'))]
print('sex / relationship mismatches:', len(mismatch))

# 3) Never-worked but reports hours
nw = df[(df.workclass == 'Never-worked') & (df['hours.per.week'] > 0)]
print('Never-worked with hours > 0:', len(nw))
nw[['workclass', 'occupation', 'hours.per.week', 'income']]

In [ ]:
# 4) capital gain and loss at the same time
print('capital.gain > 0 AND capital.loss > 0:', ((df['capital.gain'] > 0) & (df['capital.loss'] > 0)).sum())